In [3]:
import pandas as pd   
import numpy as np  

df_budget = pd.read_csv(r"C:\Users\masri\Downloads\CSV_FILES\df_budget.csv")
df_rewards = pd.read_csv(r"C:\Users\masri\Downloads\CSV_FILES\df_rewards.csv")
print(df_budget)
print(df_rewards)

           Category         Location        Date  Amount  \
0              Rent             ----  2025-01-01   -1164   
1       Restaurants  Loblaw's Market  2025-01-01    -227   
2       Restaurants        Montana's  2025-01-01     -62   
3       Restaurants  Loblaw's Market  2025-01-01    -206   
4             Gifts           Amazon  2025-01-01    -184   
...             ...              ...         ...     ...   
1032  Miscellaneous    Canadian Tire  2025-12-28    -265   
1033      Utilities             ----  2025-12-28     -52   
1034         Salary              Job  2025-12-28   10078   
1035    Restaurants        McDonalds  2025-12-28    -274   
1036      Groceries          Walmart  2025-12-28    -283   

              Payment Type  Current Balance  
0                   Cheque            70115  
1                     Visa            69888  
2                    Debit            69826  
3                   Cheque            69620  
4                     Visa            69436  
...

In [4]:
df_budget[df_budget["Amount"] < 0].groupby("Category")["Amount"].sum().abs().sort_values(ascending=False)

Category
Restaurants      104811
Rent              13968
Health            10314
Subscriptions      9433
Gifts              8740
Miscellaneous      8495
Entertainment      8191
Groceries          7490
Utilities           644
Name: Amount, dtype: int64

In [5]:
#How many times did we use a Visa card to pay for entertainment?

df_budget[
    (df_budget["Category"] == "Entertainment") &
    (df_budget["Payment Type"] == "Visa")
].shape[0]


11

In [6]:
#What card did we use the most for Restaurant purchases?
df_budget[df_budget["Category"] == "Restaurants"]["Payment Type"].value_counts().idxmax()



'Discover'

In [7]:
#How much money was spent in the category in df_budgetwith the largest expenditure? Enter your answer as a positive number.

t = df_budget["Amount"].groupby(df_budget["Category"]).sum()
t.index[t.argmin()]


'Restaurants'

In [8]:
#Which category in df_budget had the largest total expenditures?


df_budget.groupby("Category")["Amount"].sum().sort_values()


Category
Restaurants     -104811
Rent             -13968
Health           -10314
Subscriptions     -9433
Gifts             -8740
Miscellaneous     -8495
Entertainment     -8191
Groceries         -7490
Utilities          -644
Salary           120936
Name: Amount, dtype: int64

In [9]:
#drop column from ddf_rewards
df_budget = df_budget.drop(columns="Payment Method")

#re_order col exctly like the quest
df_budget = df_budget[
    ["Category", "Location", "Date", "Amount", "Payment Type", "Reward", "Current Balance"]
]

KeyError: "['Payment Method'] not found in axis"

In [ ]:
df_budget

In [ ]:
# Example: total amount spent per location
 
df_budget.groupby("Location")["Amount"].sum()


In [ ]:
# this groupby size() will tell how many times we went to location. 

print(df_budget.groupby("Location").size())
print("------------------------------")
##Probably better to sort:
print(df_budget.groupby("Location").size().sort_values(ascending=False))

In [ ]:
#Notice that size and count columns are the same.
grouped_location = df_budget[["Location", "Amount"]].groupby("Location").agg(
                    ["size", "count", "sum", "mean", "max", "min", "median"])
print(grouped_location)
print("------------------------------")

#Again sorting is better. Let's sort by sum. Notice this is a multiindex.
print(grouped_location.sort_values(("Amount","sum")))


In [10]:
#Remove the multi-index by using groupby on the series as a parameter.
#Note: Can also pass a list of series as we will see below.
grouped_location2 = df_budget["Amount"].groupby(df_budget["Location"]).agg(
                    ["size", "count", "sum", "mean", "max", "min", "median"])
print(grouped_location2)
print(grouped_location2.sort_values("sum"))

                     size  count     sum          mean    max    min   median
Location                                                                     
----                   24     24  -14612   -608.833333    -17  -1164   -630.5
A&W Canada             54     54   -9269   -171.648148    -18   -297   -170.0
Amazon                 59     59   -8740   -148.135593    -13   -300   -162.0
Best Buy               52     52   -8191   -157.519231    -16   -283   -165.5
Canadian Tire          62     62   -8495   -137.016129    -14   -269   -130.0
Domino's               60     60   -8723   -145.383333    -10   -296   -150.5
Harvey's               58     58   -9289   -160.155172    -10   -299   -176.0
Job                    12     12  120936  10078.000000  10078  10078  10078.0
Loblaw's Market        65     65  -10269   -157.984615    -10   -295   -153.0
McDonalds              60     60  -10341   -172.350000    -10   -297   -172.0
Montana's              44     44   -7301   -165.931818    -25   

In [11]:
#We can also use our own functions. 
#Suppose we wanted to isolate our spending transactions into buckets. 
#If we spend 0 to 100 we call that a small transacation.
#We call a transaction greater than 100 but less than or equal to 200 a medium transaction and then the remaining transactions we call large.
#How many of each type do we have at a given location? Let's find out!

def categorize(x: [int|float]) -> str:
    """
    Return a classification of the amount spent in x based
    on it being a small expense, a medium expense or a large expense
    
    Requires: x < 0
    
    Example:
      categorize(-13) => "small"
      categorize(-113) => "medium"
      categorize(-213) => "large"
    """
    #Recall that transactions that are debits are negative hence -x below.
    if 0 < -x <= 100:
        return "small"
    if 100 < -x <= 200:
        return "medium"
    return "large"

#Filter out only debit transactions
df_filtered = df_budget[df_budget["Amount"] < 0]

#Add a column for categorization
#Copy to avoid slice setting warning
df_loc_amount_category = df_filtered[["Location", "Amount"]].copy()
df_loc_amount_category["Categorize"] = df_filtered["Amount"].apply(categorize)

## Alternate way: Add a column for the categorization
#df_loc_amount_category = pd.concat([df_filtered[["Location", "Amount"]], 
#                           df_filtered["Amount"].apply(categorize)], 
#                           axis=1)
##Added column has the same name as "Amount" which is awkward. Rename:
#df_loc_amount_category.columns = \
#    df_loc_amount_category.columns[:-1].tolist() + ["Categorize"]

#Perform the groupby
df_grouped_categorized = df_loc_amount_category.groupby(
                            ["Location", "Categorize"]).agg("size")
print(df_grouped_categorized)
print("-----------------------------------")
# Alternatively, groupby consumes a list of series so we can just pass this.
# Notice that the column title for the Series with apply is the same as 
# the original Series which is Amount.
df_grouped_categorized2 = df_filtered.groupby([
    df_filtered["Location"], 
    df_filtered["Amount"].apply(categorize)]).agg("size")
#If desired can rename the Amount category by uncommenting the below:
#df_grouped_categorized2 = df_grouped_categorized2.rename_axis(["Location", "Categorize"])
print(df_grouped_categorized2)
print("------------------------------------")
#See only the values of the multiindex at "Walmart" using the command xs
print(df_grouped_categorized2.xs("Walmart")) 


Location             Categorize
----                 large         12
                     small         12
A&W Canada           large         23
                     medium        19
                     small         12
Amazon               large         18
                     medium        24
                     small         17
Best Buy             large         20
                     medium        13
                     small         19
Canadian Tire        large         17
                     medium        22
                     small         23
Domino's             large         14
                     medium        29
                     small         17
Harvey's             large         21
                     medium        21
                     small         16
Loblaw's Market      large         21
                     medium        22
                     small         22
McDonalds            large         25
                     medium        20
                  

In [12]:
##DataFrame has the data from df_budget in columns ["Date", "Location", "Amount"] 
#grouped by ["Date", "Location"] and we take the maximum transaction at each date and location.
df_ans = df_budget[["Date", "Location", "Amount"]].groupby(["Date", "Location"]).max()
print(df_ans)

                                Amount
Date       Location                   
2025-01-01 ----                  -1164
           A&W Canada             -160
           Amazon                 -184
           Loblaw's Market        -206
           McDonalds              -277
...                                ...
2025-12-28 Domino's               -121
           Job                   10078
           McDonalds              -274
           Shopper's Drug Mart     -26
           Walmart                -283

[952 rows x 1 columns]


In [13]:
list(df_budget.groupby("Location"))[:2]

[('----',
         Category Location        Date  Amount          Payment Type  \
  0          Rent     ----  2025-01-01   -1164                Cheque   
  76    Utilities     ----  2025-01-26     -33  Pre-Authorized Debit   
  82         Rent     ----  2025-02-01   -1164                Cheque   
  160   Utilities     ----  2025-02-25     -61  Pre-Authorized Debit   
  166        Rent     ----  2025-03-01   -1164                Cheque   
  227   Utilities     ----  2025-03-22     -89  Pre-Authorized Debit   
  256        Rent     ----  2025-04-01   -1164                Cheque   
  297   Utilities     ----  2025-04-11     -17  Pre-Authorized Debit   
  338        Rent     ----  2025-05-01   -1164                Cheque   
  406   Utilities     ----  2025-05-27     -45  Pre-Authorized Debit   
  413        Rent     ----  2025-06-01   -1164                Cheque   
  485   Utilities     ----  2025-06-23     -26  Pre-Authorized Debit   
  504        Rent     ----  2025-07-01   -1164        

In [14]:
#SERIES GROUPBY()
df_budget.groupby("Location")["Amount"].agg(["sum", "mean"])


,sum,mean
Location,,
----,-14612,-608.833333
A&W Canada,-9269,-171.648148
Amazon,-8740,-148.135593
Best Buy,-8191,-157.519231
Canadian Tire,-8495,-137.016129
Domino's,-8723,-145.383333
Harvey's,-9289,-160.155172
Job,120936,10078.000000
Loblaw's Market,-10269,-157.984615


In [15]:
#MORE GROUPBY( ) EXAMPLES

print(df_budget.groupby("Location").describe())
print(df_budget.groupby("Location")["Amount"].mean())
print(df_budget.groupby("Location")["Amount"].sum())
print(df_budget.groupby("Location").agg({"Amount":"mean", "Payment Type": pd.Series.unique}))

                    Amount                                               \
                     count          mean         std      min       25%   
Location                                                                  
----                  24.0   -608.833333  567.436315  -1164.0  -1164.00   
A&W Canada            54.0   -171.648148   80.704482   -297.0   -240.50   
Amazon                59.0   -148.135593   80.894878   -300.0   -211.00   
Best Buy              52.0   -157.519231   86.196194   -283.0   -232.00   
Canadian Tire         62.0   -137.016129   77.600172   -269.0   -207.00   
Domino's              60.0   -145.383333   78.169111   -296.0   -199.25   
Harvey's              58.0   -160.155172   82.483555   -299.0   -217.75   
Job                   12.0  10078.000000    0.000000  10078.0  10078.00   
Loblaw's Market       65.0   -157.984615   84.675943   -295.0   -227.00   
McDonalds             60.0   -172.350000   84.603727   -297.0   -239.25   
Montana's             44.

In [16]:
#Example, if we wanted to group by 2 weeks, we could pass a Grouper object pd.Grouper(key="Date", freq = "2W"). 
#For 3 months, we change freq = "2W" to freq = "3ME". Use "YE" for years and so on.
  
df_budget.Date = pd.to_datetime(df_budget.Date)
print(df_budget.groupby(pd.Grouper(key="Date", freq="3ME"))["Amount"].mean())
print("#####")
print(df_budget.groupby(pd.Grouper(key="Date", freq="3W"))["Amount"].mean())

Date
2025-01-31   -48.219512
2025-04-30   -46.996094
2025-07-31   -49.703125
2025-10-31   -40.896694
2026-01-31   -62.407960
Freq: 3ME, Name: Amount, dtype: float64
#####
Date
2025-01-05   -218.650000
2025-01-26   -158.350877
2025-02-16     26.680000
2025-03-09    -37.260274
2025-03-30     32.964286
2025-04-20   -161.096774
2025-05-11     53.851064
2025-06-01     16.823529
2025-06-22   -169.231884
2025-07-13    -17.869565
2025-08-03     18.333333
2025-08-24   -153.640625
2025-09-14     41.687500
2025-10-05     30.300000
2025-10-26   -154.229508
2025-11-16      2.310345
2025-12-07    -44.216867
2025-12-28     -5.529412
Freq: 3W-SUN, Name: Amount, dtype: float64


In [17]:
#initial_balanace is defined already - can change it if desired
df_budget.sort_values(by="Date", inplace=True, ignore_index=True)
df_budget["Current Balance"] = df_budget["Amount"].cumsum() + initial_balance
print(df_budget["Current Balance"])
print(df_budget)

NameError: name 'initial_balance' is not defined

In [18]:
#Expanding on your budget data — “How has my spending accumulated up to this date?”

df_budget["Amount"].expanding().sum()

0       -1164.0
1       -1391.0
2       -1453.0
3       -1659.0
4       -1843.0
         ...   
1032   -50455.0
1033   -50481.0
1034   -50602.0
1035   -50867.0
1036   -51150.0
Name: Amount, Length: 1037, dtype: float64

In [19]:
#Rolling on your budget data — “What is my average spending over the last N days?”

df_budget["Amount"].rolling(window=7).mean()


0               NaN
1               NaN
2               NaN
3               NaN
4               NaN
           ...     
1032    1248.142857
1033    1282.142857
1034    1301.285714
1035    1305.000000
1036    1293.857143
Name: Amount, Length: 1037, dtype: float64

In [20]:
#Shift on your budget data — “What did I spend yesterday compared to today?”

df_budget["Amount"] - df_budget["Amount"].shift(1)


0         NaN
1       937.0
2       165.0
3      -144.0
4        22.0
        ...  
1032   -222.0
1033    248.0
1034    -95.0
1035   -144.0
1036    -18.0
Name: Amount, Length: 1037, dtype: float64